In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from samos import shmodel
from datetime import datetime
from importlib import reload
import io
import cartopy.crs as ccrs
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from matplotlib.ticker import MaxNLocator
import ipywidgets as widgets
from IPython.display import display

In [ ]:
reload(shmodel)
RE = 6371.2 # km
N_data = 10000
thetas = np.linspace(0, np.pi, N_data)
phis = np.zeros_like(thetas) + 0.1
model = shmodel.SHModel.read_pygeodyn('/project/samos/2023_gji_gnm_GOVO/2023_gji_gnm_GOVO.hdf5')

## Maps of the field
Shared setup: 1° grid, Mollweide projection with coastlines (cartopy), one `calculate_field_grid` call per example.

In [ ]:
R_CMB = 3485.0                      # core-mantle boundary radius [km]
today = datetime(2020, 6, 1)

# 1-degree grid: cell edges for plotting, field evaluated at cell centres
lon_e, lat_e = np.arange(-180, 181, 1.0), np.arange(-90, 91, 1.0)
lon, lat = lon_e[:-1] + 0.5, lat_e[:-1] + 0.5
colat = 90 - lat

COMPONENTS = {                      # name: (label, colormap for absolute values)
    'Br':     (r'$B_r$',      'RdBu_r'),
    'Btheta': (r'$B_\theta$', 'RdBu_r'),
    'Bphi':   (r'$B_\phi$',   'RdBu_r'),
    'F':      (r'$F$',        'viridis'),
}
PC = ccrs.PlateCarree()


def field_maps(dates, radii):
    """All components on the grid in one model call -> dict of (Nt, Nr, Nlat, Nlon) float32 arrays."""
    Br, Bt, Bp = model.calculate_field_grid(dates, radii, colat, lon)
    F = np.sqrt(Br**2 + Bt**2 + Bp**2)
    return {k: v.astype(np.float32) for k, v in zip(COMPONENTS, (Br, Bt, Bp, F))}


def color_limits(data, diverging):
    """Symmetric limits around zero for signed quantities, min/max otherwise."""
    if diverging:
        v = float(np.abs(data).max())
        return -v, v
    return float(data.min()), float(data.max())


def new_map(ax, data, cmap, vmin, vmax):
    ax.set_global()
    ax.coastlines(linewidth = 0.6)
    ax.gridlines(linewidth = 0.3, color = 'gray', alpha = 0.5)
    return ax.pcolormesh(lon_e, lat_e, data, transform = PC, cmap = cmap,
                         vmin = vmin, vmax = vmax, shading = 'flat', rasterized = True)


def style_colorbar(cbar, label):
    """At most ~6 ticks, so large values (e.g. at the CMB) do not overlap."""
    cbar.ax.xaxis.set_major_locator(MaxNLocator(6))
    cbar.set_label(label)


def widget_map(central_lon = 0):
    """
    Off-screen figure for the interactive examples. It is not managed by pyplot,
    so it is never displayed by itself; it is rendered to PNG into an
    ipywidgets.Image (core widget, works in VS Code / JupyterLab / Colab).
    """
    fig = Figure(figsize = (9, 5.5), layout = 'constrained')
    FigureCanvasAgg(fig)
    ax = fig.add_subplot(projection = ccrs.Mollweide(central_longitude = central_lon))
    return fig, ax, widgets.Image(format = 'png')


def render(fig, image):
    buf = io.BytesIO()
    fig.savefig(buf, format = 'png', dpi = 90)
    image.value = buf.getvalue()


def zero_contour(ax, data):
    """Br = 0 line."""
    return ax.contour(lon, lat, data, levels = [0], colors = 'k', linewidths = 1.2, transform = PC)


def date_note(date):
    return ' (predicted SV)' if date > datetime(2025, 1, 1) else ''



### Example 1: Earth's magnetic field today
$B_r$, $B_\theta$ (south), $B_\phi$ (east) and the total intensity $F$ at the surface. The thick line in $B_r$ is $B_r = 0$. After 2025, IGRF-14 uses its predicted secular variation.

In [ ]:
central_lon = 0                                    # map centre longitude [deg E]

maps = field_maps(today, RE)                       # each component: (1, 1, 180, 360)

fig, axs = plt.subplots(2, 2, figsize = (12, 7), layout = 'constrained',
                        subplot_kw = {'projection': ccrs.Mollweide(central_longitude = central_lon)})

for ax, (name, (label, cmap)) in zip(axs.flat, COMPONENTS.items()):
    data = maps[name][0, 0]
    mesh = new_map(ax, data, cmap, *color_limits(data, cmap == 'RdBu_r'))
    ax.contour(lon, lat, data, levels = 10, colors = 'k', linewidths = 0.3, alpha = 0.6, transform = PC)
    if name == 'Br':
        zero_contour(ax, data)
    style_colorbar(fig.colorbar(mesh, ax = ax, orientation = 'horizontal', shrink = 0.6, pad = 0.03), f'{label} [nT]')

fig.suptitle(f'IGRF-14 at the surface, {today:%Y-%m-%d}{date_note(today)}')
plt.show()


### Example 2: Evolution from 2020 to 2025
All monthly maps are computed at once; the slider only swaps the plotted data. The colour scale is fixed over time. Tick *Change since 2020-01* to see $B(t) - B(2020)$: the secular variation is <1% of the field at the surface and hard to see otherwise.

In [ ]:
central_lon = 0                                                     # map centre longitude [deg E]

months = np.arange('2015-01', '2020-02', dtype = 'datetime64[M]')    # 2020-01 ... 2025-01 (61 maps)
maps_t = {k: v[:, 0] for k, v in field_maps(months, RE).items()}    # (61, 180, 360), one model call
change_t = {k: v - v[0] for k, v in maps_t.items()}                 # B(t) - B(2020-01)

# colour limits fixed over all dates, per component and per mode (absolute / change)
limits_t = {(name, diff): color_limits((change_t if diff else maps_t)[name], diff or cmap == 'RdBu_r')
            for name, (_, cmap) in COMPONENTS.items() for diff in (False, True)}

fig2, ax2, image2 = widget_map(central_lon)
mesh2 = new_map(ax2, maps_t['Br'][0], 'RdBu_r', *limits_t[('Br', False)])
cbar2 = fig2.colorbar(mesh2, ax = ax2, orientation = 'horizontal', shrink = 0.6, pad = 0.03)

date_slider = widgets.SelectionSlider(options = [(str(m), i) for i, m in enumerate(months)],
                                      description = 'Date', continuous_update = False, layout = widgets.Layout(width = '70%'))
component2 = widgets.Dropdown(options = list(COMPONENTS), value = 'Br', description = 'Component')
show_change = widgets.Checkbox(value = False, description = 'Change since 2020-01')
contours2 = []


def update2(_ = None):
    name, i, diff = component2.value, date_slider.value, show_change.value
    label, cmap = COMPONENTS[name]
    data = (change_t if diff else maps_t)[name][i]

    mesh2.set_array(data)                                   # swap data, no redraw from scratch
    mesh2.set_cmap('RdBu_r' if diff else cmap)
    mesh2.set_clim(*limits_t[(name, diff)])
    cbar2.update_normal(mesh2)
    style_colorbar(cbar2, f'{"Δ" if diff else ""}{label} [nT]')

    for c in contours2:
        c.remove()
    contours2.clear()
    if name == 'Br' and not diff:
        contours2.append(zero_contour(ax2, data))

    ax2.set_title(f'IGRF-14 {"change in " if diff else ""}{label} at the surface, {months[i]}')
    render(fig2, image2)


for w in (date_slider, component2, show_change):
    w.observe(update2, names = 'value')
update2()
display(widgets.VBox([widgets.HBox([component2, show_change]), date_slider, image2]))


### Example 3: From the surface down to the core-mantle boundary
All radii are computed in one call (per-degree split, so each extra radius is a weighted sum over degrees). The colour scale is rescaled at every radius, so the change in *shape* is visible: higher degrees grow faster with depth as $(a/r)^{n+2}$, revealing the flux patches at the CMB.

In [ ]:
central_lon = 0                                                     # map centre longitude [deg E]

radii = np.linspace(RE, R_CMB, 50)                                   # surface -> CMB
maps_r = {k: v[0] for k, v in field_maps(today, radii).items()}     # (50, 180, 360), one model call

fig3, ax3, image3 = widget_map(central_lon)
mesh3 = new_map(ax3, maps_r['Br'][0], 'RdBu_r', *color_limits(maps_r['Br'][0], True))
cbar3 = fig3.colorbar(mesh3, ax = ax3, orientation = 'horizontal', shrink = 0.6, pad = 0.03)

radius_slider = widgets.SelectionSlider(options = [(f'{r:.0f} km (depth {RE - r:.0f} km)', i) for i, r in enumerate(radii)],
                                        description = 'Radius', continuous_update = False, layout = widgets.Layout(width = '70%'))
component3 = widgets.Dropdown(options = list(COMPONENTS), value = 'Br', description = 'Component')
contours3 = []


def update3(_ = None):
    name, i = component3.value, radius_slider.value
    label, cmap = COMPONENTS[name]
    data = maps_r[name][i]

    mesh3.set_array(data)
    mesh3.set_cmap(cmap)
    mesh3.set_clim(*color_limits(data, cmap == 'RdBu_r'))   # rescaled at every radius
    cbar3.update_normal(mesh3)
    style_colorbar(cbar3, f'{label} [nT]')

    for c in contours3:
        c.remove()
    contours3.clear()
    if name == 'Br':
        contours3.append(zero_contour(ax3, data))

    ax3.set_title(f'IGRF-14 {label} at r = {radii[i]:.0f} km (depth {RE - radii[i]:.0f} km), '
                  f'{today:%Y-%m-%d}{date_note(today)}')
    render(fig3, image3)


for w in (radius_slider, component3):
    w.observe(update3, names = 'value')
update3()
display(widgets.VBox([component3, radius_slider, image3]))
